# Probing Metacognitive Awareness in Gemma-3: SAE-Based Analysis

**Model:** Gemma-3-4B-IT | **Three complementary approaches:**
1. **SAE Feature Analysis** — WHAT does the model represent during injection?
2. **Neurofeedback Paradigm** — CAN the model access its own injection-related representations?
3. **Attribution Patching** — WHERE (which heads/layers) does injection processing happen?

**Context:** Five prior experiments (concept vector injection, selective use, head patching) found no functional metacognitive awareness. These three SAE-based approaches provide finer-grained mechanistic evidence.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 12

RESULTS = Path('results')

# Load all result files
with open(RESULTS / 'sae_analysis' / 'feature_profiles.json') as f:
    sae_profiles = json.load(f)
with open(RESULTS / 'neurofeedback' / 'reporting_accuracy.json') as f:
    nfb_reporting = json.load(f)
with open(RESULTS / 'neurofeedback' / 'probe_results.json') as f:
    nfb_probe = json.load(f)
with open(RESULTS / 'neurofeedback' / 'control_effect.json') as f:
    nfb_control = json.load(f)
with open(RESULTS / 'attribution_patching' / 'attribution_summary.json') as f:
    attr_summary = json.load(f)
with open(RESULTS / 'cross_approach_analysis.json') as f:
    cross = json.load(f)

print('All results loaded successfully.')
print(f"SAE: {sae_profiles['n_trials']} trials, {sae_profiles['hit_rate']:.1%} hit rate")
print(f"Neurofeedback: probe 100% accuracy all layers")
print(f"Attribution: {len(attr_summary['layer_summary'])} layers scored")
print(f"\nVerdict: {cross['analysis']['verdict']}")

---
## 1. SAE Feature Analysis

For each trial (concept, layer, strength), we run clean + injected forward passes through the model, encode the residual stream through a Sparse Autoencoder (Gemma Scope 2, JumpReLU architecture), and compare which sparse features change.

- **300 trials**: 10 concepts x 3 layers x 10 repetitions
- **Layers**: 14, 22, 26 (matching Experiment 5)
- **SAEs**: Layer 22 uses standard 65k-width SAE; Layers 14/26 use 16k-width `resid_post_all` SAEs
- **Pre-filtering**: Top 200 features by activation variance across trials, then FDR-corrected statistical tests

In [ ]:
# Overview statistics
print("=" * 60)
print("SAE FEATURE ANALYSIS OVERVIEW")
print("=" * 60)
print(f"Total trials:              {sae_profiles['n_trials']}")
print(f"Hit rate:                  {sae_profiles['hit_rate']:.1%} ({int(sae_profiles['hit_rate'] * sae_profiles['n_trials'])}/{sae_profiles['n_trials']})")
print(f"Unique features seen:      {sae_profiles['n_unique_features_seen']}")
print(f"Top features (by var):     {sae_profiles['n_top_by_variance']}")
print(f"Universal features:        {sae_profiles['n_universal']}")
print(f"Concept-specific features: {sae_profiles['n_concept_specific']}")
print(f"Hit-predictive features:   {sae_profiles['n_hit_predictive']}")

### 1.1 Universal Injection-Detection Feature

Only **one** feature fires universally across concept injections: **Feature 397** (Layer 22, 65k SAE).

**Neuronpedia interpretation:** This is a *discourse initiation / formal statement marker* feature. It activates on sentence-opening tokens like "We," "The," "No," "I." It does NOT detect perturbation directly — instead, it detects the model shifting into structured response mode, which injection triggers.

This means the model has no dedicated "something was injected" detector. The closest universal signal is a side effect: injection causes the model to start generating structured responses, which activates discourse-initiation features.

In [ ]:
# Universal feature details
for feat in sae_profiles['universal_features']:
    print(f"Feature {feat['feature_id']}:")
    print(f"  Mean delta (injection - clean): {feat['mean_delta']:.1f}")
    print(f"  Fraction of concepts with positive delta: {feat['frac_concepts_positive']:.0%}")
    print(f"  Neuronpedia: 'Discourse initiation and formal statement markers'")
    print(f"  URL: https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/397")

### 1.2 Concept-Specific Features

138 features fire selectively for specific concepts. Let's visualize the top ones and check Neuronpedia for their interpretations.

In [ ]:
# Top concept-specific features
concept_feats = sae_profiles['concept_specific_features']

# Group by dominant concept
concept_groups = {}
for feat in concept_feats:
    top_concept = feat['top_concepts'][0]['concept']
    if top_concept not in concept_groups:
        concept_groups[top_concept] = []
    concept_groups[top_concept].append(feat)

# Plot: number of concept-specific features per concept
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

concepts = sorted(concept_groups.keys())
counts = [len(concept_groups[c]) for c in concepts]
colors = plt.cm.Set3(np.linspace(0, 1, len(concepts)))

ax1.barh(concepts, counts, color=colors)
ax1.set_xlabel('Number of Concept-Specific SAE Features')
ax1.set_title('Concept-Specific Features by Concept')
for i, v in enumerate(counts):
    ax1.text(v + 0.5, i, str(v), va='center', fontsize=10)

# Plot: mean delta of top concept-specific features
top_20 = sorted(concept_feats, key=lambda x: abs(x['mean_delta']), reverse=True)[:20]
feat_ids = [f"{f['feature_id']}\n({f['top_concepts'][0]['concept']})" for f in top_20]
deltas = [f['mean_delta'] for f in top_20]
bar_colors = ['#d32f2f' if d < 0 else '#1976d2' for d in deltas]

ax2.barh(range(len(feat_ids)), deltas, color=bar_colors)
ax2.set_yticks(range(len(feat_ids)))
ax2.set_yticklabels(feat_ids, fontsize=8)
ax2.set_xlabel('Mean Activation Delta (injection - clean)')
ax2.set_title('Top 20 Concept-Specific Features by |Delta|')
ax2.axvline(0, color='black', linewidth=0.5)
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('results/sae_concept_specific_features.png', bbox_inches='tight')
plt.show()

print(f"\nTotal concept-specific features: {len(concept_feats)}")
print(f"Concepts represented: {', '.join(concepts)}")

#### Neuronpedia Lookups: Concept-Specific Feature Mismatches

Interesting finding: many concept-specific features don't match their injected concept semantically.

| Feature | Injected Concept | Neuronpedia Interpretation | Match? |
|---------|-----------------|---------------------------|--------|
| 33328 | guitar | "Guitar-related content" | Yes |
| 3163 | rain | "Meditation, mindfulness, contemplative wellness" | No |
| 12383 | fire | "With additions or accompaniments" | No |
| 12517 | mountain | "Street and address names" | No |
| 33574 | cat | "Possessive determiners, definite articles" | No |
| 332 | (suppressed) | "Educational/explanatory exposition" | N/A |

**Interpretation:** Concept vectors don't cleanly align with SAE feature directions. Injection activates features along *incidental* dimensions of the concept vector, not along semantically matching features. This is consistent with injection being a crude perturbation rather than a targeted semantic manipulation.

*Note: Only Layer 22 (65k) features are on Neuronpedia. Layers 14/26 use 16k `resid_post_all` SAEs which are not hosted.*

Sources:
- [Feature 33328 (Guitar)](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/33328)
- [Feature 3163 (Meditation)](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/3163)
- [Feature 12383 (Accompaniment)](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/12383)
- [Feature 12517 (Addresses)](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/12517)
- [Feature 332 (Exposition)](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/332)

### 1.3 Hit-Predictive Features

83 features whose activation delta significantly predicts whether the model will behaviorally detect the injection (name the concept in its response). These are the most metacognitively relevant features.

In [ ]:
# Hit-predictive features analysis
hit_feats = sae_profiles['hit_predictive_features']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Cohen's d distribution
ds = [f['cohens_d'] for f in hit_feats]
axes[0].hist(ds, bins=30, color='#5c6bc0', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='black', linewidth=0.5, linestyle='--')
axes[0].set_xlabel("Cohen's d")
axes[0].set_ylabel('Count')
axes[0].set_title(f"Hit-Predictive Features (n={len(hit_feats)})\nCohen's d Distribution")

# Plot 2: Top 15 by |Cohen's d|
top_15 = sorted(hit_feats, key=lambda x: abs(x['cohens_d']), reverse=True)[:15]
feat_labels = [str(f['feature_id']) for f in top_15]
d_vals = [f['cohens_d'] for f in top_15]
bar_colors = ['#d32f2f' if d < 0 else '#388e3c' for d in d_vals]

axes[1].barh(range(len(feat_labels)), d_vals, color=bar_colors)
axes[1].set_yticks(range(len(feat_labels)))
axes[1].set_yticklabels(feat_labels, fontsize=9)
axes[1].set_xlabel("Cohen's d")
axes[1].set_title("Top 15 Hit-Predictive Features")
axes[1].axvline(0, color='black', linewidth=0.5)
axes[1].invert_yaxis()

# Add legend
pos_patch = mpatches.Patch(color='#388e3c', label='Activated by hits')
neg_patch = mpatches.Patch(color='#d32f2f', label='Suppressed by hits')
axes[1].legend(handles=[pos_patch, neg_patch], fontsize=8, loc='lower right')

# Plot 3: Mean activation in hits vs misses for top features
top_10 = sorted(hit_feats, key=lambda x: abs(x['cohens_d']), reverse=True)[:10]
feat_labels_10 = [str(f['feature_id']) for f in top_10]
mean_hits = [abs(f['mean_hit']) for f in top_10]
mean_misses = [abs(f['mean_miss']) for f in top_10]

x = np.arange(len(feat_labels_10))
width = 0.35
axes[2].bar(x - width/2, mean_hits, width, label='Hit trials', color='#388e3c', alpha=0.8)
axes[2].bar(x + width/2, mean_misses, width, label='Miss trials', color='#9e9e9e', alpha=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(feat_labels_10, rotation=45, fontsize=8)
axes[2].set_ylabel('|Mean Activation Delta|')
axes[2].set_title('Hit vs Miss Activation (Top 10)')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('results/sae_hit_predictive_features.png', bbox_inches='tight')
plt.show()

print("\nTop 5 hit-predictive features:")
for f in top_15[:5]:
    direction = 'ACTIVATED' if f['cohens_d'] > 0 else 'SUPPRESSED'
    print(f"  Feature {f['feature_id']}: d={f['cohens_d']:+.3f}, "
          f"hit={f['mean_hit']:.1f}, miss={f['mean_miss']:.1f} [{direction}]")

#### Key Hit-Predictive Features & Neuronpedia Interpretations

| Feature | Cohen's d | Hit Mean | Miss Mean | Neuronpedia | Metacognitive Role |
|---------|----------|----------|-----------|-------------|--------------------|
| **870** | +1.263 | 83.6 | 0.0 | *Not on Neuronpedia (L14 16k)* | Fires ONLY during hits; strongest predictor |
| **7165** | +1.254 | 67.3 | 0.0 | *Not on Neuronpedia (L14 16k)* | Binary: on for hits, off for misses |
| **147** | +1.176 | 107.9 | 23.6 | *Not on Neuronpedia (L14 16k)* | Fires weakly during misses too — graded signal |
| **5715** | -1.017 | -48.1 | +6.1 | "Surprise / astonishment" | **SUPPRESSED** during hits — model exits surprise mode |
| **254** | -0.984 | -100.2 | -23.3 | *L14 16k* | Strongly suppressed during hits |
| **41871** | +0.848 | 42.4 | 0.0 | "Yes/No determinations" | Fires when model outputs "Yes" / "Detected" |
| **33328** | +0.848 | 42.4 | 0.0 | "Guitar-related content" | Genuine concept feature (guitar injection) |

**Key insight:** Feature 5715 ("surprise/astonishment") is *suppressed* during successful detection. When injection successfully causes the model to name a concept, the model shifts from surprise mode to confident assertion mode. This is not metacognition — it's a downstream effect of the injection being strong enough to override the model's default behavior.

Feature 41871 ("Yes/No determinations") is mechanistically meaningful: it fires when the model produces "Yes" / "Detected" responses to the Exp1 prompt. This is a *response feature*, not a *detection feature*.

Sources:
- [Feature 5715 (Surprise)](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/5715)
- [Feature 41871 (Yes/Detected)](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/41871)

---
## 2. Neurofeedback Paradigm

Adapted from [Bai et al. (2505.13763)](https://arxiv.org/abs/2505.13763):
1. **Train a linear probe** (logistic regression) to distinguish injected vs. clean activations
2. **Feed probe labels back** to the model via in-context learning (ICL)
3. **Test reporting**: Can the model report its own injection state given ICL examples?
4. **Test control**: Does ICL labeling shift the model's internal representations along the probe axis?

Labels: `0` = no injection, `1` = injection present (single tokens for clean logit comparison)

### 2.1 Probe Accuracy (Sanity Check)

The linear probe should trivially distinguish injected vs. clean activations — injection IS a real perturbation. This confirms our injection is working.

In [ ]:
# Probe results
print("Linear Probe Results (Injection Detection)")
print("=" * 50)

fig, ax = plt.subplots(figsize=(8, 4))

layers = sorted(nfb_probe.keys(), key=int)
train_accs = [nfb_probe[l]['train_acc'] for l in layers]
test_accs = [nfb_probe[l]['test_acc'] for l in layers]
axis_norms = [nfb_probe[l]['lr_axis_norm'] for l in layers]

x = np.arange(len(layers))
width = 0.3

bars1 = ax.bar(x - width/2, train_accs, width, label='Train', color='#1976d2', alpha=0.8)
bars2 = ax.bar(x + width/2, test_accs, width, label='Test', color='#388e3c', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([f'Layer {l}' for l in layers])
ax.set_ylabel('Accuracy')
ax.set_ylim(0.9, 1.02)
ax.set_title('Linear Probe: Injection Detection Accuracy')
ax.legend()
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Chance')

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.0%}', ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.0%}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('results/neurofeedback_probe_accuracy.png', bbox_inches='tight')
plt.show()

print("\nProbe axis norms (larger = more separable):")
for l in layers:
    print(f"  Layer {l}: ||w|| = {nfb_probe[l]['lr_axis_norm']:.4f}, "
          f"train={nfb_probe[l]['train_acc']:.0%}, test={nfb_probe[l]['test_acc']:.0%}")
print("\n100% probe accuracy at all layers confirms injection is a real, detectable perturbation.")
print("Layer 14 has the largest axis norm (0.183) — injection is most distinctive there.")

### 2.2 Reporting Accuracy: Can the Model Report Its Own State?

The critical test: given ICL examples mapping injection states to labels (0/1), can the model correctly classify new trials?

We measure `logit("1") - logit("0")` at the last token position, varying the number of ICL examples from 0 to 64.

In [ ]:
# Reporting accuracy vs number of ICL examples
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

layer_colors = {'14': '#d32f2f', '22': '#1976d2', '26': '#388e3c'}

for layer in sorted(nfb_reporting.keys(), key=int):
    n_contexts = sorted(nfb_reporting[layer].keys(), key=int)
    accs = [nfb_reporting[layer][n]['accuracy'] for n in n_contexts]
    ax1.plot([int(n) for n in n_contexts], accs,
             marker='o', linewidth=2, markersize=6,
             label=f'Layer {layer}', color=layer_colors[layer])

ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
ax1.set_xlabel('Number of ICL Examples')
ax1.set_ylabel('Reporting Accuracy')
ax1.set_title('Can the Model Report Its Injection State?')
ax1.legend()
ax1.set_ylim(0.4, 1.0)
ax1.set_xticks([0, 4, 8, 16, 32, 64])
ax1.grid(alpha=0.3)

# Plot 2: Logit difference (injected vs clean)
for layer in sorted(nfb_reporting.keys(), key=int):
    n_contexts = sorted(nfb_reporting[layer].keys(), key=int)
    inject_logits = [nfb_reporting[layer][n]['mean_logit_diff_injected'] for n in n_contexts]
    clean_logits = [nfb_reporting[layer][n]['mean_logit_diff_clean'] for n in n_contexts]

    ns = [int(n) for n in n_contexts]
    ax2.plot(ns, inject_logits, marker='o', linewidth=2,
             label=f'L{layer} injected', color=layer_colors[layer])
    ax2.plot(ns, clean_logits, marker='s', linewidth=1, linestyle='--',
             label=f'L{layer} clean', color=layer_colors[layer], alpha=0.5)

ax2.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax2.set_xlabel('Number of ICL Examples')
ax2.set_ylabel('Mean logit("1") - logit("0")')
ax2.set_title('Logit Difference by Condition')
ax2.legend(fontsize=8, ncol=2)
ax2.set_xticks([0, 4, 8, 16, 32, 64])
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/neurofeedback_reporting_accuracy.png', bbox_inches='tight')
plt.show()

print("Key finding: Accuracy rises from ~50% (chance) to 88% with 64 ICL examples.")
print("But is this metacognition or ICL pattern-matching?")
print("\nNotice: all 3 layers converge to 88% at n=64, suggesting the model")
print("learns a generic input-output mapping, not layer-specific self-monitoring.")

### 2.3 Control Effect: Does ICL Labeling Shift Internal Representations?

The strongest test of metacognition: in a 2x2 design (ICL_label=0/1) x (injection=on/off), do ICL labels cause the model to shift its representations along the probe axis?

If the model has metacognitive *control*, then being told "you are in state 1" should push representations toward the injection direction (even without injection).

In [ ]:
# Control effect: 2x2 design
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, layer in enumerate(sorted(nfb_control.keys(), key=int)):
    ax = axes[idx]
    ctrl = nfb_control[layer]
    cm = ctrl['condition_means']
    cs = ctrl['condition_stds']

    conditions = ['No Inject\nLabel=0', 'No Inject\nLabel=1',
                  'Inject\nLabel=0', 'Inject\nLabel=1']
    means = [cm['label0_noinject'], cm['label1_noinject'],
             cm['label0_inject'], cm['label1_inject']]
    stds = [cs['label0_noinject'], cs['label1_noinject'],
            cs['label0_inject'], cs['label1_inject']]

    colors = ['#90caf9', '#42a5f5', '#ef9a9a', '#e53935']
    bars = ax.bar(range(4), means, yerr=stds, capsize=4,
                  color=colors, edgecolor='white', alpha=0.85)
    ax.set_xticks(range(4))
    ax.set_xticklabels(conditions, fontsize=8)
    ax.set_ylabel('Projection onto LR Axis')
    ax.set_title(f"Layer {layer}\nCohen's d = {ctrl['cohens_d']:.3f}")
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')

plt.suptitle('Control Effect: Does ICL Labeling Shift Representations?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('results/neurofeedback_control_effect.png', bbox_inches='tight')
plt.show()

print("Control Effect (Cohen's d):")
for layer in sorted(nfb_control.keys(), key=int):
    d = nfb_control[layer]['cohens_d']
    interpretation = 'negligible' if abs(d) < 0.2 else ('small' if abs(d) < 0.5 else 'medium')
    print(f"  Layer {layer}: d = {d:+.3f} ({interpretation})")

print("\nThe key comparison is Label=0 vs Label=1 WITHIN each injection condition.")
print("The tiny Cohen's d values (< 0.32) mean ICL labels barely shift representations.")
print("The model cannot CONTROL its internal state based on metacognitive labels.")
print("\nNotice the HUGE gap between inject/no-inject (red vs blue bars) —")
print("injection is a massive perturbation, but the label has almost no effect.")

### 2.4 Neurofeedback Summary

| Layer | Probe Acc | Best Report Acc | Best n_context | Control d | Conclusion |
|-------|-----------|-----------------|----------------|-----------|------------|
| 14 | 100% | 88% | 64 | +0.079 | Can report, cannot control |
| 22 | 100% | 88% | 64 | +0.067 | Can report, cannot control |
| 26 | 100% | 88% | 64 | -0.315 | Can report, cannot control |

**Interpretation for metacognition:**
- **Representation**: YES — the probe trivially separates injected from clean (100% accuracy)
- **Access (reporting)**: PARTIAL — the model reaches 88% with many ICL examples, but this may be ICL pattern-matching rather than introspection. All layers converge to the same accuracy, suggesting a generic capability
- **Control**: NO — ICL labels do not meaningfully shift internal representations (Cohen's d < 0.32). The model cannot use metacognitive information to modify its own state

---
## 3. Attribution Patching

Gradient-based approximation to identify which layers and attention heads are most important for processing the injection signal:

$$\text{effect}_i = (h^{\text{clean}}_i - h^{\text{corrupt}}_i) \cdot \nabla_{h^{\text{corrupt}}_i} \mathcal{L}$$

- 90 trials: 10 concepts x 3 injection layers x 3 strengths
- Metric: logit of 'yes' token in Exp1 detection prompt

In [ ]:
# Layer-level attribution scores
layer_summary = attr_summary['layer_summary']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9))

# Plot 1: All layers
layer_ids = sorted(layer_summary.keys(), key=int)
mean_scores = [layer_summary[l]['mean_abs_score'] for l in layer_ids]
max_scores = [layer_summary[l]['max_abs_score'] for l in layer_ids]

# Color by score intensity
norm_scores = np.array(mean_scores)
norm_scores = norm_scores / max(norm_scores) if max(norm_scores) > 0 else norm_scores
colors = plt.cm.YlOrRd(norm_scores)

ax1.bar([int(l) for l in layer_ids], mean_scores, color=colors, edgecolor='white', alpha=0.9)
ax1.set_xlabel('Layer')
ax1.set_ylabel('Mean |Attribution Score|')
ax1.set_title('Layer-Level Attribution: Where Does Injection Processing Happen?')
ax1.set_xticks(range(0, 34, 2))
ax1.grid(axis='y', alpha=0.3)

# Mark injection layers
for inj_layer in [14, 22, 26]:
    ax1.axvline(inj_layer, color='blue', linestyle=':', alpha=0.4)
    ax1.text(inj_layer, max(mean_scores) * 0.95, f'inj L{inj_layer}',
             ha='center', fontsize=7, color='blue')

# Mark global attention layers
for ga_layer in [5, 11, 17, 23, 29]:
    ax1.plot(ga_layer, mean_scores[ga_layer] + 0.3, 'v', color='purple', markersize=6)

ga_patch = mpatches.Patch(color='purple', label='Global attention layers')
inj_patch = mpatches.Patch(color='blue', label='Injection layers')
ax1.legend(handles=[ga_patch, inj_patch], fontsize=9)

# Plot 2: Top 10 layers
top_layers = attr_summary['top_layers']
tl_ids = [f"L{t['layer']}" for t in top_layers]
tl_means = [t['mean_abs_score'] for t in top_layers]
tl_maxs = [t['max_abs_score'] for t in top_layers]

x = np.arange(len(tl_ids))
ax2.bar(x - 0.2, tl_means, 0.4, label='Mean |score|', color='#ff7043')
ax2.bar(x + 0.2, tl_maxs, 0.4, label='Max |score|', color='#ffb74d', alpha=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(tl_ids)
ax2.set_ylabel('Attribution Score')
ax2.set_title('Top 10 Layers by Attribution Score')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/attribution_layer_scores.png', bbox_inches='tight')
plt.show()

print("Top 5 layers by attribution:")
for t in top_layers[:5]:
    print(f"  Layer {t['layer']}: mean={t['mean_abs_score']:.2f}, max={t['max_abs_score']:.1f}")
print("\nKey: Layers 0-13 have ZERO attribution (below injection point).")
print("Late layers (26-32) dominate — injection propagates forward.")

### 3.2 Head-Level Attribution

Which attention heads within the most important layers are driving injection processing?

In [ ]:
# Head-level attribution heatmap
head_summary = attr_summary['head_summary']
head_layers = sorted(head_summary.keys(), key=int)
n_heads = 8

# Build heatmap matrix
heatmap = np.zeros((len(head_layers), n_heads))
for i, layer in enumerate(head_layers):
    for h in range(n_heads):
        heatmap[i, h] = head_summary[layer][str(h)]['mean_abs_score']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = ax1.imshow(heatmap, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax1.set_xticks(range(n_heads))
ax1.set_xticklabels([f'H{h}' for h in range(n_heads)])
ax1.set_yticks(range(len(head_layers)))
ax1.set_yticklabels([f'Layer {l}' for l in head_layers])
ax1.set_title('Head Attribution Scores')
ax1.set_xlabel('Head')

# Add value annotations
for i in range(len(head_layers)):
    for j in range(n_heads):
        val = heatmap[i, j]
        color = 'white' if val > heatmap.max() * 0.6 else 'black'
        ax1.text(j, i, f'{val:.2f}', ha='center', va='center',
                 fontsize=8, color=color)

plt.colorbar(im, ax=ax1, label='Mean |Attribution|')

# Bar chart of top heads across all layers
all_heads = []
for layer in head_layers:
    for h in range(n_heads):
        score = head_summary[layer][str(h)]['mean_abs_score']
        all_heads.append((f'L{layer}H{h}', score))

all_heads.sort(key=lambda x: x[1], reverse=True)
top_heads = all_heads[:10]

labels = [h[0] for h in top_heads]
scores = [h[1] for h in top_heads]

ax2.barh(range(len(labels)), scores, color='#ff7043')
ax2.set_yticks(range(len(labels)))
ax2.set_yticklabels(labels)
ax2.set_xlabel('Mean |Attribution Score|')
ax2.set_title('Top 10 Attention Heads')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('results/attribution_head_scores.png', bbox_inches='tight')
plt.show()

print("Top 5 heads by attribution score:")
for label, score in top_heads[:5]:
    print(f"  {label}: {score:.3f}")
print("\nLayer 15 dominates head-level attribution — it's the first layer")
print("AFTER injection at L14 (the most sensitive injection layer).")
print("Heads 7 and 6 in L15 do most of the immediate injection processing.")

---
## 4. Cross-Approach Synthesis: What Does This Tell Us About Metacognition?

The three approaches test different components of metacognitive awareness:

```
Attribution Patching → identifies circuit components (WHERE)
        ↓
SAE Feature Analysis → decomposes what those components represent (WHAT)
        ↓
Neurofeedback → tests if model can access/act on those representations (HOW)
```

In [ ]:
# Cross-approach summary visualization
fig, ax = plt.subplots(figsize=(12, 6))

# Three components of metacognition
components = ['Representation\n(SAE)', 'Access\n(Neurofeedback\nReporting)', 'Control\n(Neurofeedback\nControl)']
evidence = [1.0, 0.88, 0.08]  # representation=yes, access=88%, control=~0
colors_comp = ['#4caf50', '#ff9800', '#f44336']
labels_comp = ['Strong evidence', 'Partial evidence', 'No evidence']

bars = ax.bar(range(3), evidence, color=colors_comp, edgecolor='white',
              width=0.6, alpha=0.85)

# Add text labels
details = [
    '1 universal feature\n138 concept-specific\n83 hit-predictive',
    '88% reporting accuracy\nat n=64 ICL examples\n(50% at n=0)',
    "Cohen's d < 0.32\nICL labels don't shift\ninternal representations"
]
for i, (bar, detail) in enumerate(zip(bars, details)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            detail, ha='center', va='bottom', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_xticks(range(3))
ax.set_xticklabels(components, fontsize=12)
ax.set_ylabel('Evidence Strength', fontsize=12)
ax.set_ylim(0, 1.5)
ax.set_title('Three Components of Metacognitive Awareness\n'
             'Gemma-3-4B-IT', fontsize=14)

# Add threshold lines
ax.axhline(0.5, color='orange', linestyle='--', alpha=0.5)
ax.text(2.8, 0.52, 'Chance', fontsize=9, color='orange')

plt.tight_layout()
plt.savefig('results/metacognition_synthesis.png', bbox_inches='tight')
plt.show()

In [ ]:
# Print the verdict
analysis = cross['analysis']

print("=" * 70)
print("CROSS-APPROACH VERDICT")
print("=" * 70)
print()
for finding in analysis['findings']:
    print(f"  {finding}")
print()
print(f"  Representation: {'YES' if analysis['has_representation'] else 'NO'}")
print(f"  Access:         {'YES' if analysis['has_access'] else 'NO'}")
print(f"  Control:        {'YES' if analysis['has_control'] else 'NO'}")
print()
print(f"  VERDICT: {analysis['verdict']}")
print()
print("=" * 70)

## 5. Mechanistic Picture: The Metacognition That Isn't

### What happens when we inject a concept vector?

**WHERE:** Attribution patching shows injection effects propagate forward through layers 14-33, with the strongest processing in the final layers (26-32). Layer 15's heads H7 and H6 do the initial processing immediately after the L14 injection point.

**WHAT:** SAE decomposition reveals:
- **One universal feature** (397: "discourse initiation") detects the model shifting into response mode — not injection itself
- **Concept-specific features** often don't match their concept semantically ("rain" activates meditation features, "mountain" activates address features). Injection doesn't cleanly activate semantic features — it perturbs along incidental dimensions
- **Hit-predictive features** include response markers ("Yes/Detected" feature 41871) and surprise suppressors (feature 5715 decreases during hits). These track the *behavioral consequence* of injection, not an internal detection mechanism

**HOW:** The neurofeedback paradigm shows:
- A linear probe trivially detects injection (100% accuracy) — the information EXISTS in the representation
- The model can REPORT its injection state with 88% accuracy given enough ICL examples — it can ACCESS the information
- But ICL labels do NOT shift internal representations (Cohen's d < 0.32) — the model cannot CONTROL its state based on metacognitive information

### The Gap Between Representation and Control

This is the critical finding. The model satisfies the first two requirements for metacognition:
1. It *represents* injection state (probe accuracy = 100%)
2. It can *access/report* that state (reporting accuracy = 88%)

But it fails the third:
3. It **cannot use metacognitive information to modify its own processing**

This matches our earlier experiments:
- Exp4 showed behavioral control (think/don't-think prompts) doesn't shift internal representations
- Exp5 showed the model can't selectively use or resist injections based on relevance
- The neurofeedback control test confirms this at a mechanistic level: even when explicitly told its state, the model's representations don't move

### What "Metacognition" Would Look Like

If the model had genuine metacognitive awareness, we would expect:
- SAE features that specifically detect perturbation (not just response-mode features)
- Neurofeedback control Cohen's d > 0.5 (ICL labels shifting representations)
- Attribution showing a dedicated monitoring circuit (not just forward propagation)
- Concept-specific features matching their injected concepts (clean semantic encoding)

Instead, what we see is a model that processes injection as generic perturbation — it propagates forward, activates incidental features, and sometimes produces concept-relevant output, but there is no internal monitoring, evaluation, or control mechanism.

---
## 6. Neuropedia Feature Reference

### Layer 22 (65k SAE) Features on Neuronpedia

| Feature | Role in Our Analysis | Neuronpedia Description | URL |
|---------|---------------------|------------------------|-----|
| 397 | Universal injection detector | Discourse initiation, formal statement markers | [Link](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/397) |
| 5715 | Hit-predictive (suppressed) | Surprise / astonishment | [Link](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/5715) |
| 33328 | Hit-predictive (guitar) | Guitar-related content | [Link](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/33328) |
| 41871 | Hit-predictive (response) | Yes/No determinations | [Link](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/41871) |
| 332 | Suppressed by injection | Educational/explanatory exposition | [Link](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/332) |
| 3163 | Concept-specific (rain) | Meditation, mindfulness practices | [Link](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/3163) |
| 12383 | Concept-specific (fire) | With additions or accompaniments | [Link](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/12383) |
| 12517 | Concept-specific (mountain) | Street and address names | [Link](https://www.neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-65k/12517) |

### Layer 14 (16k `resid_post_all` SAE) — NOT on Neuronpedia

The three strongest hit-predictive features (870, 7165, 147) are from layer 14's 16k SAE, which is not hosted on Neuronpedia. These features are exclusively active during hit trials (mean_miss = 0.0 for 870 and 7165), making them potential markers of when injection strength exceeds the model's "absorption threshold."

To interpret these features, one would need to examine their decoder weight vectors directly using the locally cached SAE weights and find their top-activating tokens across a corpus.

In [ ]:
# Summary statistics table
print("\n" + "=" * 70)
print("SUMMARY: SAE-Based Metacognition Analysis — Gemma-3-4B-IT")
print("=" * 70)
print()
print("Approach            | Key Metric              | Value")
print("-" * 70)
print(f"SAE Features        | Universal detectors     | {sae_profiles['n_universal']}")
print(f"                    | Concept-specific feats  | {sae_profiles['n_concept_specific']}")
print(f"                    | Hit-predictive feats    | {sae_profiles['n_hit_predictive']}")
print(f"                    | Top Cohen's d           | {max(abs(f['cohens_d']) for f in sae_profiles['hit_predictive_features']):.3f}")
print(f"Neurofeedback       | Probe accuracy          | 100% (all layers)")
print(f"                    | Best reporting acc      | 88% (n=64)")
print(f"                    | Control Cohen's d       | {max(abs(nfb_control[l]['cohens_d']) for l in nfb_control):.3f}")
print(f"Attribution         | Top layer               | L{attr_summary['top_layers'][0]['layer']} (score={attr_summary['top_layers'][0]['mean_abs_score']:.2f})")
print(f"                    | Top head                | L15H7 (score=1.968)")
print("-" * 70)
print(f"VERDICT: {cross['analysis']['verdict']}")
print()
print("The model REPRESENTS and can ACCESS its injection state,")
print("but CANNOT USE that information to control its own processing.")
print("This is the hallmark of a system without metacognitive awareness.")

---
## 7. Gemma-3-27B-IT Results (Cross-Model Comparison)

We ran the same three approaches on the 27B model (2x A100 GPUs, `device_map="auto"`):
- **Attribution Patching**: COMPLETE (gradient-free variant for multi-GPU)
- **Neurofeedback**: COMPLETE (L25, L40, L47 — all layers, with reporting and control)
- **SAE Feature Analysis**: COMPLETE (300 trials, detection prompt)

### Key question: Does scale (4B -> 27B) change the metacognition picture?

In [ ]:
# Load 27B results (all complete)
RESULTS_27B = Path('results_27b')

with open(RESULTS_27B / 'attribution_patching' / 'attribution_summary.json') as f:
    attr_27b = json.load(f)
with open(RESULTS_27B / 'neurofeedback' / 'reporting_accuracy.json') as f:
    nfb_27b_reporting = json.load(f)
with open(RESULTS_27B / 'neurofeedback' / 'probe_results.json') as f:
    nfb_27b_probe = json.load(f)
with open(RESULTS_27B / 'neurofeedback' / 'control_effect.json') as f:
    nfb_27b_control = json.load(f)
with open(RESULTS_27B / 'sae_analysis' / 'feature_profiles.json') as f:
    sae_27b_profiles = json.load(f)

print("27B results loaded (all complete).")
print(f"  Attribution: {len(attr_27b['layer_summary'])} layers (62 total, gradient-free)")
print(f"  Neurofeedback: {len(nfb_27b_reporting)} layers reporting, "
      f"{len(nfb_27b_probe)} layers probed, {len(nfb_27b_control)} layers control")
print(f"  SAE: {sae_27b_profiles['n_trials']} trials, "
      f"{sae_27b_profiles['hit_rate']:.1%} hit rate, "
      f"{sae_27b_profiles['n_hit_predictive']} hit-predictive features")

### 7.1 Attribution Patching (27B) — Gradient-Free

The 27B model spans 2 GPUs, so we used the gradient-free approximation:
$$\text{score}_i = \|h^{\text{clean}}_i - h^{\text{corrupt}}_i\|^2 \times (\mathcal{L}_{\text{clean}} - \mathcal{L}_{\text{corrupt}})$$

**Important:** Absolute scores are not directly comparable to 4B (different method, larger hidden dim = 5376 vs 2560). We compare the *relative pattern* across layers.

In [ ]:
# 27B Attribution: Layer-level scores
layer_summary_27b = attr_27b['layer_summary']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: 27B layer attribution (normalized to show relative pattern)
layer_ids_27b = sorted(layer_summary_27b.keys(), key=int)
mean_scores_27b = np.array([layer_summary_27b[l]['mean_abs_score'] for l in layer_ids_27b])

# Normalize to [0, 1] for comparison
max_27b = mean_scores_27b.max()
norm_27b = mean_scores_27b / max_27b if max_27b > 0 else mean_scores_27b

colors_27b = plt.cm.YlOrRd(norm_27b)
ax1.bar([int(l) for l in layer_ids_27b], norm_27b, color=colors_27b, edgecolor='white', alpha=0.9)
ax1.set_xlabel('Layer')
ax1.set_ylabel('Normalized Attribution Score')
ax1.set_title('27B Layer Attribution (62 layers)\nGradient-Free Method')
ax1.set_xticks(range(0, 62, 5))
ax1.grid(axis='y', alpha=0.3)

# Mark injection layers
for inj_layer in [25, 40, 47]:
    ax1.axvline(inj_layer, color='blue', linestyle=':', alpha=0.4)
    ax1.text(inj_layer, 1.02, f'inj L{inj_layer}', ha='center', fontsize=7, color='blue')

# Plot 2: Side-by-side comparison with 4B (both normalized)
mean_scores_4b = np.array([attr_summary['layer_summary'][l]['mean_abs_score']
                           for l in sorted(attr_summary['layer_summary'].keys(), key=int)])
max_4b = mean_scores_4b.max()
norm_4b = mean_scores_4b / max_4b if max_4b > 0 else mean_scores_4b

# Normalize layer positions to % of total depth
pct_4b = np.linspace(0, 100, len(norm_4b))
pct_27b = np.linspace(0, 100, len(norm_27b))

ax2.plot(pct_4b, norm_4b, 'o-', color='#1976d2', linewidth=2, markersize=3,
         label='4B (34 layers)', alpha=0.8)
ax2.plot(pct_27b, norm_27b, 's-', color='#d32f2f', linewidth=2, markersize=2,
         label='27B (62 layers)', alpha=0.8)
ax2.set_xlabel('Layer Position (% of Total Depth)')
ax2.set_ylabel('Normalized Attribution Score')
ax2.set_title('4B vs 27B: Attribution by Relative Depth')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/attribution_27b_comparison.png', bbox_inches='tight')
plt.show()

print("Both models show the same pattern: attribution increases monotonically")
print("from injection point to final layer. No dedicated 'monitoring circuit' exists —")
print("injection simply propagates forward through the residual stream.")
print(f"\n27B top layer: L{attr_27b['top_layers'][0]['layer']} "
      f"(last layer, score={attr_27b['top_layers'][0]['mean_abs_score']:.0f})")
print("This monotonic increase is even cleaner in 27B — consistent with")
print("injection being a passive perturbation, not an actively monitored signal.")

### 7.2 Neurofeedback (27B) — Full Results

All three layers (L25, L40, L47) completed for reporting, probe, and control.

**Striking finding:** The 27B reporting accuracy curve is even more binary than 4B -- it stays at exactly 50% (chance) from n=0 through n=32, then jumps to 92% at n=64. This "all-or-nothing" pattern suggests the model isn't gradually learning to introspect; instead, at n=64 it has enough ICL examples for pure pattern-matching.

In [ ]:
# 27B vs 4B Neurofeedback comparison (now with full 27B data from JSON)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: 4B reporting accuracy (all 3 layers)
layer_colors_4b = {'14': '#d32f2f', '22': '#1976d2', '26': '#388e3c'}
for layer in sorted(nfb_reporting.keys(), key=int):
    n_contexts = sorted(nfb_reporting[layer].keys(), key=int)
    accs = [nfb_reporting[layer][n]['accuracy'] for n in n_contexts]
    ax1.plot([int(n) for n in n_contexts], accs,
             marker='o', linewidth=2, markersize=6,
             label=f'4B L{layer}', color=layer_colors_4b[layer])

# Add 27B results (all 3 layers from JSON)
layer_colors_27b = {'25': '#ff6f00', '40': '#6a1b9a', '47': '#00695c'}
for layer in sorted(nfb_27b_reporting.keys(), key=int):
    n_contexts = sorted(nfb_27b_reporting[layer].keys(), key=int)
    accs = [nfb_27b_reporting[layer][n]['accuracy'] for n in n_contexts]
    ax1.plot([int(n) for n in n_contexts], accs,
             marker='s', linewidth=2, markersize=6, linestyle='--',
             label=f'27B L{layer}', color=layer_colors_27b[layer])

ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('Number of ICL Examples')
ax1.set_ylabel('Reporting Accuracy')
ax1.set_title('Reporting Accuracy: 4B vs 27B')
ax1.legend(fontsize=8, ncol=2)
ax1.set_ylim(0.4, 1.0)
ax1.set_xticks([0, 4, 8, 16, 32, 64])
ax1.grid(alpha=0.3)

# Plot 2: Control effect comparison (now with all 27B layers)
control_data = {}
for layer in sorted(nfb_control.keys(), key=int):
    control_data[f'4B L{layer}'] = nfb_control[layer]['cohens_d']
for layer in sorted(nfb_27b_control.keys(), key=int):
    control_data[f'27B L{layer}'] = nfb_27b_control[layer]['cohens_d']

labels = list(control_data.keys())
d_vals = list(control_data.values())
colors_ctrl = ['#d32f2f', '#1976d2', '#388e3c'] + ['#ff6f00', '#6a1b9a', '#00695c'][:len(nfb_27b_control)]

ax2.bar(range(len(labels)), d_vals, color=colors_ctrl[:len(labels)], alpha=0.8, edgecolor='white')
ax2.set_xticks(range(len(labels)))
ax2.set_xticklabels(labels, fontsize=9)
ax2.set_ylabel("Cohen's d")
ax2.set_title('Control Effect: 4B vs 27B')
ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax2.axhline(0.5, color='red', linewidth=1, linestyle=':', alpha=0.5)
ax2.text(len(labels) - 0.5, 0.52, 'small\neffect', fontsize=8, color='red', alpha=0.7)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/neurofeedback_27b_comparison.png', bbox_inches='tight')
plt.show()

print("27B Neurofeedback Summary:")
for layer in sorted(nfb_27b_probe.keys(), key=int):
    print(f"  L{layer}: probe train={nfb_27b_probe[layer]['train_acc']:.0%}, "
          f"test={nfb_27b_probe[layer]['test_acc']:.0%}")
for layer in sorted(nfb_27b_reporting.keys(), key=int):
    best_n = max(nfb_27b_reporting[layer].keys(), key=lambda n: nfb_27b_reporting[layer][n]['accuracy'])
    best_acc = nfb_27b_reporting[layer][best_n]['accuracy']
    print(f"  L{layer}: best reporting={best_acc:.0%} (n={best_n})")
for layer in sorted(nfb_27b_control.keys(), key=int):
    print(f"  L{layer}: control d={nfb_27b_control[layer]['cohens_d']:.3f}")
print()
print("The 27B model shows the SAME pattern as 4B:")
print("  - Represents injection (probe=100%)")
print("  - Can report with enough ICL (92%)")
print("  - Cannot control internal state (Cohen's d negligible)")
print("  - Step-function reporting curve argues FOR pattern-matching")

### 7.3 SAE Feature Analysis (27B) — Detection Prompt

The 27B SAE analysis completed with 300 trials using the detection prompt (Exp1).

| Metric | 4B | 27B |
|--------|------|------|
| Hit rate | 23.3% | 6.7% |
| Universal features | 1 (F397) | 0 |
| Concept-specific | 138 | 77 |
| Hit-predictive | 83 | 52 |
| Top Cohen's d | 1.26 | 5.53 |

**Key differences:**
- **27B has 0 universal features** -- no feature fires across all concepts. The 4B's F397 ("discourse initiation") was prompt-specific behavior that the 27B doesn't exhibit
- **Much lower hit rate** (6.7% vs 23.3%) -- the 27B is far more robust to injection at 5% strength, consistent with Exp5 findings
- **Extremely strong hit-predictive features** in 27B -- Feature 174 has d=5.53, far exceeding anything in 4B. But with only 20 hits in 300 trials, these features are essentially "does the output garble" markers rather than metacognitive detectors

### 7.4 Cross-Model Verdict

| Component | 4B | 27B | Scale Effect |
|-----------|------|------|-------------|
| **Representation** (probe) | 100% | 100% | None -- both trivially detectable |
| **Access** (reporting @ n=64) | 88% | 92% | Slight improvement, but step-function pattern |
| **Control** (Cohen's d) | 0.08-0.31 | negligible | None -- both negligible |
| **Attribution pattern** | Monotonic increase L14->L33 | Monotonic increase L25->L61 | Same passive forward propagation |
| **SAE hit rate** | 23.3% | 6.7% | 27B far more robust to injection |
| **Universal features** | 1 (F397) | 0 | 27B doesn't even shift response mode |

**Conclusion: Scale does not create metacognitive awareness.** The 27B model shows the exact same dissociation as 4B: injection state is represented and (with enough ICL) reportable, but the model cannot use that information to control its own processing. The 27B's greater robustness to perturbation (from Exp5) is not metacognition -- it's the same indifference expressed as absorption rather than disruption.

---
## 8. Detection vs Neutral Prompt: Unmasking the Confound

### The Problem

Sections 1-6 used the **Exp1 detection prompt**, which tells the model:
> "You are an AI. Sometimes researchers inject a 'thought'... Do you detect an injected thought?"

This means any SAE features we found could be reacting to the **unusual prompt** rather than to injection itself. Feature 5715 ("surprise/astonishment") being suppressed during hits might just reflect the model finding the detection question less surprising when injection overrides its output.

### The Fix

We re-ran the SAE analysis with **neutral prompts** -- 50 everyday sentences (e.g., "The temperature dropped significantly overnight") with no mention of injection. A "hit" in neutral mode means the injected concept word **leaked** into the model's response to an unrelated sentence.

This gives us the cleanest test: do the same SAE features change during injection even when the model has no reason to think about injection?

In [ ]:
# Load neutral prompt results for both models
with open(RESULTS / 'sae_analysis' / 'feature_profiles_neutral.json') as f:
    sae_4b_neutral = json.load(f)
with open(RESULTS_27B / 'sae_analysis' / 'feature_profiles_neutral.json') as f:
    sae_27b_neutral = json.load(f)

# Rename for clarity
sae_4b_detection = sae_profiles  # already loaded
sae_27b_detection = sae_27b_profiles  # already loaded

# Summary table
print("=" * 75)
print("SAE FEATURE ANALYSIS: DETECTION vs NEUTRAL PROMPT")
print("=" * 75)
print()
print(f"{'Metric':<30} {'4B Det':>8} {'4B Neut':>8} {'27B Det':>8} {'27B Neut':>8}")
print("-" * 75)
for metric, key in [
    ('Hit rate', 'hit_rate'),
    ('Universal features', 'n_universal'),
    ('Concept-specific', 'n_concept_specific'),
    ('Hit-predictive', 'n_hit_predictive'),
    ('Unique features seen', 'n_unique_features_seen'),
]:
    vals = [sae_4b_detection[key], sae_4b_neutral[key],
            sae_27b_detection[key], sae_27b_neutral[key]]
    if key == 'hit_rate':
        print(f"{metric:<30} {vals[0]:>7.1%} {vals[1]:>7.1%} {vals[2]:>7.1%} {vals[3]:>7.1%}")
    else:
        print(f"{metric:<30} {vals[0]:>8} {vals[1]:>8} {vals[2]:>8} {vals[3]:>8}")
print("-" * 75)

In [ ]:
# Visualization: Detection vs Neutral comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: Hit rates across conditions ---
ax = axes[0, 0]
conditions = ['4B\nDetection', '4B\nNeutral', '27B\nDetection', '27B\nNeutral']
hit_rates = [sae_4b_detection['hit_rate'], sae_4b_neutral['hit_rate'],
             sae_27b_detection['hit_rate'], sae_27b_neutral['hit_rate']]
bar_colors = ['#1976d2', '#64b5f6', '#d32f2f', '#ef9a9a']
bars = ax.bar(range(4), [h * 100 for h in hit_rates], color=bar_colors, edgecolor='white')
ax.set_xticks(range(4))
ax.set_xticklabels(conditions)
ax.set_ylabel('Hit Rate (%)')
ax.set_title('Concept Detection / Leakage Rate')
for bar, rate in zip(bars, hit_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{rate:.1%}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, 30)

# --- Plot 2: Feature counts comparison ---
ax = axes[0, 1]
metrics = ['Universal', 'Concept-\nspecific', 'Hit-\npredictive']
x = np.arange(len(metrics))
width = 0.2
datasets = [
    ('4B Det', [sae_4b_detection['n_universal'], sae_4b_detection['n_concept_specific'],
                sae_4b_detection['n_hit_predictive']], '#1976d2'),
    ('4B Neut', [sae_4b_neutral['n_universal'], sae_4b_neutral['n_concept_specific'],
                 sae_4b_neutral['n_hit_predictive']], '#64b5f6'),
    ('27B Det', [sae_27b_detection['n_universal'], sae_27b_detection['n_concept_specific'],
                 sae_27b_detection['n_hit_predictive']], '#d32f2f'),
    ('27B Neut', [sae_27b_neutral['n_universal'], sae_27b_neutral['n_concept_specific'],
                  sae_27b_neutral['n_hit_predictive']], '#ef9a9a'),
]
for i, (label, vals, color) in enumerate(datasets):
    ax.bar(x + i * width - 1.5 * width, vals, width, label=label, color=color, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Count')
ax.set_title('Feature Counts by Category')
ax.legend(fontsize=8)

# --- Plot 3: 4B detection vs neutral hit-predictive features ---
ax = axes[1, 0]
det_feat_ids = {f['feature_id'] for f in sae_4b_detection['hit_predictive_features']}
neut_feat_ids = {f['feature_id'] for f in sae_4b_neutral['hit_predictive_features']}
overlap = det_feat_ids & neut_feat_ids
det_only = det_feat_ids - neut_feat_ids
neut_only = neut_feat_ids - det_feat_ids

# Simple Venn-like bar chart
categories = ['Detection\nonly', 'Both', 'Neutral\nonly']
counts_venn = [len(det_only), len(overlap), len(neut_only)]
colors_venn = ['#1976d2', '#7b1fa2', '#64b5f6']
bars = ax.bar(range(3), counts_venn, color=colors_venn, edgecolor='white')
ax.set_xticks(range(3))
ax.set_xticklabels(categories)
ax.set_ylabel('Number of Hit-Predictive Features')
ax.set_title('4B: Feature Overlap (Detection vs Neutral)')
for bar, count in zip(bars, counts_venn):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', fontsize=12, fontweight='bold')

# Highlight key confounded features
confounded = [5715, 397]
confound_in_det = sum(1 for f in confounded if f in det_feat_ids)
confound_in_neut = sum(1 for f in confounded if f in neut_feat_ids)
ax.text(0.5, 0.95, f'F5715 (surprise): detection only\nF397 (discourse): detection only',
        transform=ax.transAxes, fontsize=8, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# --- Plot 4: Top hit-predictive features side-by-side ---
ax = axes[1, 1]
# Get top 10 from each
top_det = sorted(sae_4b_detection['hit_predictive_features'],
                 key=lambda x: abs(x['cohens_d']), reverse=True)[:8]
top_neut = sorted(sae_4b_neutral['hit_predictive_features'],
                  key=lambda x: abs(x['cohens_d']), reverse=True)[:8]

y_det = np.arange(len(top_det))
y_neut = np.arange(len(top_neut)) + len(top_det) + 1

det_ds = [f['cohens_d'] for f in top_det]
neut_ds = [f['cohens_d'] for f in top_neut]
det_labels = [f"F{f['feature_id']}" for f in top_det]
neut_labels = [f"F{f['feature_id']}" for f in top_neut]

ax.barh(y_det, det_ds, color=['#d32f2f' if d < 0 else '#1976d2' for d in det_ds], alpha=0.8)
ax.barh(y_neut, neut_ds, color=['#d32f2f' if d < 0 else '#64b5f6' for d in neut_ds], alpha=0.8)

ax.set_yticks(list(y_det) + list(y_neut))
ax.set_yticklabels(det_labels + neut_labels, fontsize=8)
ax.axhline(len(top_det) + 0.5, color='black', linewidth=2, linestyle='-')
ax.text(0.02, 0.75, 'DETECTION', transform=ax.transAxes, fontsize=9, fontweight='bold', color='#1976d2')
ax.text(0.02, 0.25, 'NEUTRAL', transform=ax.transAxes, fontsize=9, fontweight='bold', color='#64b5f6')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel("Cohen's d")
ax.set_title("4B: Top Hit-Predictive Features")
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('results/detection_vs_neutral_comparison.png', bbox_inches='tight')
plt.show()

print(f"\n4B Hit-predictive feature overlap:")
print(f"  Detection only: {len(det_only)} features")
print(f"  Overlap: {len(overlap)} features")
print(f"  Neutral only: {len(neut_only)} features")
if overlap:
    print(f"  Shared features: {sorted(overlap)}")

### 8.1 The Surprise Feature Was a Confound

**Feature 5715 ("surprise/astonishment")** was one of the most interesting findings from the detection prompt analysis -- it was the 4th strongest hit-predictive feature (d=-1.02), and its suppression during hits suggested a shift from surprise to confident assertion.

**But it was entirely driven by the detection prompt.**

| Condition | F5715 appearances | Role |
|-----------|------------------|------|
| 4B Detection | 100/300 trials (Layer 22 only) | #4 hit-predictive (d=-1.02) |
| 4B Neutral | **0/300 trials** | Completely absent |

Feature 5715 doesn't appear in a single neutral trial. It wasn't detecting injection -- it was reacting to the unusual Exp1 prompt ("You are an AI. Sometimes researchers inject a 'thought'..."). The model finds this question surprising at baseline, and that surprise gets overridden when injection is strong enough to force a confident "yes" response.

Similarly, **Feature 397 ("discourse initiation")** -- the ONLY universal feature in detection mode -- drops to just 7/300 appearances in neutral mode and loses its universal status. It was tracking the model's shift into structured response format, not injection per se.

### 8.2 What Survives the Confound Check?

The 4B neutral analysis reveals **genuine** injection effects:
- **17.7% concept leakage** -- even without asking about injection, the model spontaneously mentions the injected concept 17.7% of the time. Injection really does contaminate output
- **Feature 39** replaces F397 as the only universal feature -- this is a different, potentially more genuine perturbation signal
- **45 hit-predictive features** (vs 83 in detection) -- fewer but potentially more meaningful, since they predict concept leakage without any prompt cuing
- **Feature 1221** (d=2.21) is by far the strongest hit-predictive feature in neutral mode, much stronger than anything in detection mode. This feature specifically tracks when injection is strong enough to leak into unrelated output

In [ ]:
# Sample neutral-mode responses that leaked concepts
with open(RESULTS / 'sae_analysis' / 'sae_feature_scan_neutral.json') as f:
    neutral_trials = json.load(f)

hits_neutral = [t for t in neutral_trials if t['hit']]
misses_neutral = [t for t in neutral_trials if not t['hit']]

print("=" * 70)
print(f"4B NEUTRAL: {len(hits_neutral)} hits / {len(neutral_trials)} trials ({len(hits_neutral)/len(neutral_trials):.1%})")
print("=" * 70)

# Concept breakdown
from collections import Counter
concept_hits = Counter(t['concept'] for t in hits_neutral)
concept_total = Counter(t['concept'] for t in neutral_trials)
print("\nHit rate by concept:")
for concept in sorted(concept_total.keys()):
    h = concept_hits.get(concept, 0)
    t = concept_total[concept]
    print(f"  {concept:>10}: {h:>2}/{t} ({h/t:.0%})")

print("\nSample hit responses (concept leaked into neutral prompt response):")
for t in hits_neutral[:5]:
    resp = t['response'][:150].replace('\n', ' ')
    print(f"  [{t['concept']}/L{t['layer']}] {resp}...")

### 8.3 The 27B Is Completely Impervious

The most striking result is from the 27B neutral analysis:

| Model | Neutral Hit Rate | Meaning |
|-------|-----------------|---------|
| 4B | 17.7% (53/300) | Injection leaks concepts into ~1 in 6 responses |
| 27B | **0.0% (0/300)** | Injection NEVER leaks concepts into responses |

The 27B model at 5% injection strength is so robust that the injected concept never contaminates its output on neutral prompts. This confirms what Exp5 showed: the 27B absorbs perturbation rather than being deflected by it. The perturbation exists in the residual stream (probes detect it at 100%), but the model's downstream processing is completely indifferent to it.

This has direct implications:
- **0 hit-predictive features** in 27B neutral (no hits = nothing to predict)
- **0 universal features** (injection doesn't consistently shift any feature direction)
- The 78 "concept-specific" features have all-zero deltas (noise from the feature selection pipeline)

The 27B model's larger representational capacity acts as a buffer: the same 5% perturbation that causes concept leakage in 4B is absorbed without any behavioral consequence in 27B.

### 8.4 Revised Interpretation: What the SAE Analysis Actually Shows

The detection vs neutral comparison fundamentally changes the interpretation:

**What we thought (detection prompt only):**
- Feature 5715 ("surprise") was suppressed during hits -> model shifts from surprise to confidence
- Feature 397 ("discourse initiation") was a universal injection detector
- 83 hit-predictive features provided a rich signature of injection processing

**What we now know (after neutral prompt control):**
- Feature 5715 was reacting to the weird Exp1 prompt, not to injection
- Feature 397 was a response-format feature, not an injection detector
- Many hit-predictive features from detection mode were tracking prompt-driven response patterns, not injection

**What genuinely survives:**
- Injection really does leak concepts (17.7% in 4B neutral) -- this is not a prompt artifact
- Different hit-predictive features emerge in neutral mode, tracking genuine concept leakage
- Feature 39 (universal in neutral) and Feature 1221 (top hit-predictive, d=2.21) may represent authentic perturbation-detection signals
- Scale provides robustness (27B: 0% leakage) but not metacognition

**Bottom line:** The "surprise" feature finding that initially seemed most metacognitively interesting was entirely an artifact of telling the model to look for injection. When we remove that prompt cue, the model has no surprise feature, no discourse-initiation feature -- just generic perturbation effects that sometimes cause concept words to leak into output.

In [ ]:
# Final comprehensive summary
print("=" * 75)
print("FINAL SUMMARY: SAE-Based Metacognition Analysis")
print("=" * 75)
print()
print("                          4B Detection  4B Neutral  27B Detection  27B Neutral")
print("-" * 75)
print(f"  Hit rate                    {sae_4b_detection['hit_rate']:>7.1%}      {sae_4b_neutral['hit_rate']:>7.1%}        {sae_27b_detection['hit_rate']:>7.1%}        {sae_27b_neutral['hit_rate']:>7.1%}")
print(f"  Universal features          {sae_4b_detection['n_universal']:>7}      {sae_4b_neutral['n_universal']:>7}        {sae_27b_detection['n_universal']:>7}        {sae_27b_neutral['n_universal']:>7}")
print(f"  Hit-predictive features     {sae_4b_detection['n_hit_predictive']:>7}      {sae_4b_neutral['n_hit_predictive']:>7}        {sae_27b_detection['n_hit_predictive']:>7}        {sae_27b_neutral['n_hit_predictive']:>7}")
print(f"  F5715 (surprise) present?   {'YES':>7}      {'NO':>7}        {'N/A':>7}        {'N/A':>7}")
print(f"  F397 (discourse) universal? {'YES':>7}      {'NO':>7}        {'NO':>7}        {'NO':>7}")
print("-" * 75)
print()
print("KEY FINDINGS:")
print("  1. Feature 5715 ('surprise') was ENTIRELY a prompt confound (0/300 neutral trials)")
print("  2. Feature 397 ('discourse') was MOSTLY a prompt confound (7/300 neutral, not universal)")
print("  3. 4B genuinely leaks concepts 17.7% of the time even with neutral prompts")
print("  4. 27B NEVER leaks concepts at 5% strength (0/300 neutral trials)")
print("  5. Different hit-predictive features emerge in neutral mode (F1221 d=2.21)")
print()
print("CONCLUSION:")
print("  Neither model has metacognitive awareness. The original 'surprise' finding")
print("  was an artifact of the detection prompt. After removing the confound,")
print("  injection is just perturbation: it sometimes leaks concepts (4B) or gets")
print("  absorbed entirely (27B), but neither model monitors, evaluates, or")
print("  selectively responds to the injection.")
print("=" * 75)